# VOYARA — Validation Deep Learning : classification des demandes de voyage
**Boutheina Rjab** · Transfer Learning vs Fine-Tuning de `xlm-roberta-base`

Tâche : à partir du texte d'une demande, prédire `type_voyage` (famille / couple / amis / solo / non_precise) et les besoins de transport (`vol`, `voiture` : oui / non / indifferent).

Progression mesurée sur **le même jeu de test (115 demandes, jamais utilisé pour entraîner)** :
1. ML : TF-IDF + régression logistique
2. DL Transfer Learning : XLM-R **gelé** + régression logistique
3. DL Fine-Tuning : XLM-R **dégelé** (phase 1 : 4 dernières couches, phase 2 : tout)

> Colab : Exécution → Modifier le type d'exécution → **GPU T4**.

## 0. Installation et données

In [ ]:
!git clone -q https://github.com/ALI0benjannet/voyara.git
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import os
from google.colab import files
os.makedirs("voyara/services/ai/data/dl", exist_ok=True)
files.upload()   # choisis classif_train.jsonl
os.replace("classif_train.jsonl", "voyara/services/ai/data/dl/classif_train.jsonl")

Saving classif_train.jsonl to classif_train.jsonl


In [ ]:
import json, os, random, time
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import set_seed
SEED = 42; set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; print("Appareil :", DEVICE)

EVAL = "voyara/services/ai/data/eval/nlu_demandes.jsonl"
TRAIN_AUG = "voyara/services/ai/data/dl/classif_train.jsonl"     # dev + paraphrases (étape 1)
rows = [json.loads(l) for l in open(EVAL, encoding="utf-8")]

def label(r, champ):
    a = r["attendu"]
    if champ == "type_voyage":
        t = a["type_voyage"]
        return "amis" if t == "groupe" else (t or "non_precise")   # 'groupe' : 3 exemples -> fusionné avec 'amis'
    return a["besoins_transport"][champ]

def depuis_demande(r):
    return {"origine": r["id"], "texte": r["texte"], "langue": r["attendu"]["langue"],
            **{c: label(r, c) for c in ["type_voyage", "vol", "voiture"]}}

dev  = [depuis_demande(r) for r in rows if r["jeu"] == "dev"]
test = [depuis_demande(r) for r in rows if r["jeu"] == "test"]   # NE SERT QU'À L'ÉVALUATION FINALE
train = [json.loads(l) for l in open(TRAIN_AUG, encoding="utf-8")] if os.path.exists(TRAIN_AUG) else dev
print(f"train = {len(train)} exemples ({'augmenté' if train is not dev else 'dev seul, sans paraphrases'}) · test = {len(test)}")
pd.Series([e["type_voyage"] for e in train]).value_counts()

Appareil : cuda
train = 308 exemples (augmenté) · test = 115


,count
famille,88
couple,76
amis,64
non_precise,56
solo,24


In [ ]:
CHAMPS = ["type_voyage", "vol", "voiture"]
LANGUES = ["fr", "en", "ar", "darija"]
resultats = []          # une ligne par (approche, champ)

def evaluer(nom, champ, y_pred, duree_s=None):
    y = [e[champ] for e in test]
    ligne = {"approche": nom, "champ": champ, "F1_macro": round(f1_score(y, y_pred, average="macro"), 3)}
    for lg in LANGUES:
        idx = [i for i, e in enumerate(test) if e["langue"] == lg]
        ligne[f"F1_{lg}"] = round(f1_score([y[i] for i in idx], [y_pred[i] for i in idx], average="macro"), 3)
    ligne["entrainement_s"] = duree_s
    resultats.append(ligne); print(ligne)

## 1. ML — TF-IDF + régression logistique (référence)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
for champ in CHAMPS:
    t0 = time.time()
    ml = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)),
                       LogisticRegression(max_iter=2000, class_weight="balanced"))
    ml.fit([e["texte"] for e in train], [e[champ] for e in train])
    evaluer("ML TF-IDF + LogReg", champ, list(ml.predict([e["texte"] for e in test])), round(time.time() - t0, 1))

{'approche': 'ML TF-IDF + LogReg', 'champ': 'type_voyage', 'F1_macro': 0.46, 'F1_fr': 0.47, 'F1_en': 0.314, 'F1_ar': 0.407, 'F1_darija': 0.462, 'entrainement_s': 0.6}
{'approche': 'ML TF-IDF + LogReg', 'champ': 'vol', 'F1_macro': 0.513, 'F1_fr': 0.507, 'F1_en': 0.375, 'F1_ar': 0.574, 'F1_darija': 0.642, 'entrainement_s': 0.4}
{'approche': 'ML TF-IDF + LogReg', 'champ': 'voiture', 'F1_macro': 0.604, 'F1_fr': 0.666, 'F1_en': 0.619, 'F1_ar': 0.467, 'F1_darija': 0.577, 'entrainement_s': 0.4}


## 2. DL — Transfer Learning : XLM-R gelé + tête légère
**Point 1 (modèle)** : `xlm-roberta-base` (278 M), pré-entraîné sur 100 langues dont l'arabe → adapté au mélange fr / en / ar / darija. Écartés : CamemBERT (français seul), AraBERT (arabe seul), mBERT (moins bon que XLM-R).

**Point 2 (Transfer Learning)** : les poids de XLM-R ne bougent pas ; on extrait un vecteur par demande (mean pooling) et on entraîne seulement une régression logistique dessus.

In [ ]:
from transformers import AutoTokenizer, AutoModel
MODELE = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODELE)
enc = AutoModel.from_pretrained(MODELE).to(DEVICE).eval()

@torch.no_grad()
def vecteurs(textes, bs=32):
    out = []
    for i in range(0, len(textes), bs):
        b = tok(textes[i:i+bs], padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
        h = enc(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.vstack(out)

t0 = time.time()
V_tr, V_te = vecteurs([e["texte"] for e in train]), vecteurs([e["texte"] for e in test])
t_vec = time.time() - t0
for champ in CHAMPS:
    tete = LogisticRegression(max_iter=5000, class_weight="balanced").fit(V_tr, [e[champ] for e in train])
    evaluer("DL Transfer Learning (XLM-R gelé)", champ, list(tete.predict(V_te)), round(t_vec, 1))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'approche': 'DL Transfer Learning (XLM-R gelé)', 'champ': 'type_voyage', 'F1_macro': 0.366, 'F1_fr': 0.41, 'F1_en': 0.157, 'F1_ar': 0.299, 'F1_darija': 0.372, 'entrainement_s': 1.3}
{'approche': 'DL Transfer Learning (XLM-R gelé)', 'champ': 'vol', 'F1_macro': 0.595, 'F1_fr': 0.685, 'F1_en': 0.571, 'F1_ar': 0.435, 'F1_darija': 0.743, 'entrainement_s': 1.3}
{'approche': 'DL Transfer Learning (XLM-R gelé)', 'champ': 'voiture', 'F1_macro': 0.538, 'F1_fr': 0.606, 'F1_en': 0.822, 'F1_ar': 0.451, 'F1_darija': 0.329, 'entrainement_s': 1.3}


## 3. DL — Fine-Tuning (dégel progressif)
**Point 3 (stratégie)** : phase 1 = seules les 4 dernières couches + la tête sont entraînées (les couches basses gardent leurs connaissances générales, moins de sur-apprentissage avec peu de données) ; phase 2 = tout le modèle, avec un taux d'apprentissage plus faible. Validation = 15 % des **demandes d'origine** (les paraphrases d'une même demande restent ensemble), early stopping sur le F1.

In [ ]:
import sys; sys.last_traceback = None
gc.collect(); torch.cuda.empty_cache()

In [ ]:
from transformers import TrainerCallback

class GarderMeilleur(TrainerCallback):
    """Garde le meilleur modèle en RAM (pas d'écriture disque) + early stopping."""
    def __init__(self, patience=5): self.best, self.etat, self.attente, self.patience = -1, None, 0, patience
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kw):
        if metrics["eval_f1"] > self.best:
            self.best, self.attente = metrics["eval_f1"], 0
            self.etat = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.attente += 1
            if self.attente >= self.patience: control.should_training_stop = True

def fine_tuner(champ, couches_gelees, lr, epochs=15, sauver=False):
    classes = sorted({e[champ] for e in train}); c2i = {c: i for i, c in enumerate(classes)}
    n = np.array([sum(e[champ] == c for e in tr_part) for c in classes], dtype=float)
    poids = torch.tensor(n.sum() / (len(classes) * np.maximum(n, 1)), dtype=torch.float)
    def ds(ex):
        d = Dataset.from_dict({"text": [e["texte"] for e in ex], "label": [c2i[e[champ]] for e in ex]})
        return d.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODELE, num_labels=len(classes))
    if couches_gelees:
        for p in model.roberta.embeddings.parameters(): p.requires_grad = False
        for couche in model.roberta.encoder.layer[:couches_gelees]:
            for p in couche.parameters(): p.requires_grad = False
    args = TrainingArguments("ckpt", learning_rate=lr, num_train_epochs=epochs, per_device_train_batch_size=16,
            eval_strategy="epoch", save_strategy="no", seed=SEED, report_to="none",
            logging_strategy="epoch", fp16=True)
    f1 = lambda p: {"f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}
    cb = GarderMeilleur(patience=5)
    tr = TrainerPondere(model=model, args=args, train_dataset=ds(tr_part), eval_dataset=ds(va_part),
                        processing_class=tok, compute_metrics=f1, poids=poids, callbacks=[cb])
    t0 = time.time(); tr.train(); duree = round(time.time() - t0, 1)
    model.load_state_dict(cb.etat)                            # meilleur modèle selon la validation
    pred = [classes[i] for i in tr.predict(ds(test)).predictions.argmax(-1)]
    historique = tr.state.log_history
    if sauver: tr.save_model(f"modeles/xlmr-voyara-{champ}")   # une seule écriture à la fin
    del tr, model; gc.collect(); torch.cuda.empty_cache()
    return pred, duree, historique

for champ in ["vol", "voiture"]:                              # type_voyage déjà fait
    y1, d1, h1 = fine_tuner(champ, couches_gelees=8, lr=3e-5, sauver=True)
    evaluer("DL Fine-Tuning phase 1 (4 couches)", champ, y1, d1)
    ft[champ], courbes[champ] = (y1,), (h1,)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,1.123975,1.099725,0.102564
2,1.093630,1.062603,0.280702
3,1.083533,1.054834,0.402646
4,0.963213,0.914195,0.514550
5,0.723353,0.721650,0.693244
6,0.533616,0.623924,0.822222
7,0.385465,0.568547,0.813600
8,0.270799,0.503156,0.776075
9,0.194095,0.470045,0.795712
10,0.174482,0.522718,0.765722


Map:   0%|          | 0/115 [00:00<?, ? examples/s]

... (6 outputs left out)

Epoch,Training Loss,Validation Loss,F1
1,1.105273,1.040533,0.280702
2,1.116750,1.039937,0.280702
3,1.101877,1.040260,0.359420
4,1.106198,1.042114,0.326599
5,1.043908,1.081645,0.478607
6,0.997170,0.868211,0.407018
7,0.829346,0.653993,0.713854
8,0.624546,0.455414,0.895330
9,0.448432,0.418402,0.782906
10,0.337135,0.385362,0.784127


Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'approche': 'DL Fine-Tuning phase 1 (4 couches)', 'champ': 'voiture', 'F1_macro': 0.769, 'F1_fr': 0.845, 'F1_en': 1.0, 'F1_ar': 0.881, 'F1_darija': 0.252, 'entrainement_s': 16.4}


In [ ]:
y, d, h = fine_tuner("type_voyage", couches_gelees=8, lr=3e-5, sauver=True)
ft["type_voyage"], courbes["type_voyage"] = (y,), (h,)
print("F1 macro type_voyage (modèle sauvegardé) :", round(f1_score([e["type_voyage"] for e in test], y, average="macro"), 3))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,1.626114,1.612698,0.085714
2,1.617171,1.637161,0.085714
3,1.591350,1.662308,0.124571
4,1.582012,1.792654,0.157233
5,1.577727,1.618658,0.173617
6,1.407773,1.348054,0.516537
7,1.194079,1.277175,0.472066
8,0.968873,0.966154,0.683223
9,0.766655,0.896366,0.663132
10,0.645155,0.913096,0.672222


Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

F1 macro type_voyage (modèle sauvegardé) : 0.505


## 4. Résultats (point 4)

In [ ]:
df_tout = pd.DataFrame(resultats)
print(df_tout[["approche", "champ", "F1_macro"]])            # historique complet (tentatives comprises)

df = df_tout.drop_duplicates(subset=["approche", "champ"], keep="last")   # dernier run = version finale
os.makedirs("resultats", exist_ok=True)
df_tout.to_csv("resultats/dl_classif_historique.csv", index=False)       # garde les tentatives pour l'analyse
df.to_csv("resultats/dl_classif.csv", index=False)
open("resultats/dl_classif.md", "w").write(df.to_markdown(index=False))

ordre = ["ML TF-IDF + LogReg", "DL Transfer Learning (XLM-R gelé)",
         "DL Fine-Tuning phase 1 (4 couches)", "DL Fine-Tuning phase 2 (tout)"]
df.pivot(index="approche", columns="champ", values="F1_macro").reindex([o for o in ordre if o in set(df["approche"])])

                              approche        champ  F1_macro
0                   ML TF-IDF + LogReg  type_voyage     0.460
1                   ML TF-IDF + LogReg          vol     0.513
2                   ML TF-IDF + LogReg      voiture     0.604
3    DL Transfer Learning (XLM-R gelé)  type_voyage     0.366
4    DL Transfer Learning (XLM-R gelé)          vol     0.595
5    DL Transfer Learning (XLM-R gelé)      voiture     0.538
6   DL Fine-Tuning phase 1 (4 couches)  type_voyage     0.489
7        DL Fine-Tuning phase 2 (tout)  type_voyage     0.362
8   DL Fine-Tuning phase 1 (4 couches)          vol     0.530
9        DL Fine-Tuning phase 2 (tout)          vol     0.287
10  DL Fine-Tuning phase 1 (4 couches)      voiture     0.277
11  DL Fine-Tuning phase 1 (4 couches)  type_voyage     0.600
12       DL Fine-Tuning phase 2 (tout)  type_voyage     0.486
13  DL Fine-Tuning phase 1 (4 couches)          vol     0.644
14  DL Fine-Tuning phase 1 (4 couches)      voiture     0.769


champ,type_voyage,voiture,vol
approche,,,
ML TF-IDF + LogReg,0.460,0.604,0.513
DL Transfer Learning (XLM-R gelé),0.366,0.538,0.595
DL Fine-Tuning phase 1 (4 couches),0.600,0.769,0.644
DL Fine-Tuning phase 2 (tout),0.486,NaN,0.287


In [ ]:
# Matrice de confusion du meilleur modèle pour type_voyage
y = [e["type_voyage"] for e in test]; yp = ft["type_voyage"][0]
classes = sorted(set(y) | set(yp))
print(classification_report(y, yp, zero_division=0))
pd.DataFrame(confusion_matrix(y, yp, labels=classes), index=classes, columns=classes)

              precision    recall  f1-score   support

        amis       0.45      0.56      0.50        25
      couple       0.65      0.65      0.65        23
     famille       0.67      0.83      0.74        29
 non_precise       0.36      0.26      0.30        19
        solo       0.45      0.26      0.33        19

    accuracy                           0.55       115
   macro avg       0.52      0.51      0.51       115
weighted avg       0.53      0.55      0.53       115



,amis,couple,famille,non_precise,solo
amis,14,4,4,3,0
couple,2,15,5,0,1
famille,2,2,24,0,1
non_precise,7,0,3,5,4
solo,6,2,0,6,5


In [ ]:
scores = {c: [] for c in CHAMPS}
for s in [42, 7, 2026]:
    SEED = s; set_seed(s)
    for c in CHAMPS:
        y, _, _ = fine_tuner(c, couches_gelees=8, lr=3e-5)          # sans sauvegarde
        scores[c].append(f1_score([e[c] for e in test], y, average="macro"))
SEED = 42; set_seed(SEED)
pd.DataFrame({c: [f"{np.mean(v):.3f} ± {np.std(v):.3f}"] for c, v in scores.items()},
             index=["DL Fine-Tuning phase 1 — 3 graines"])

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/264 [00:00<?, ? examples/s]

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,1.633175,1.556653,0.085714
2,1.630443,1.800592,0.156576
3,1.584987,1.715045,0.283810
4,1.477001,1.587222,0.358596
5,1.243059,1.228961,0.572424
6,1.000386,1.001254,0.669371
7,0.824384,1.122067,0.523773
8,0.680388,0.842141,0.702244
9,0.551181,0.751677,0.637845
10,0.474017,0.880413,0.610000


Map:   0%|          | 0/115 [00:00<?, ? examples/s]

... (45 outputs left out)

Map:   0%|          | 0/44 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1
1,1.127937,1.028573,0.266667
2,1.113631,1.087704,0.148059
3,1.102840,1.080197,0.240870
4,1.081735,1.062795,0.275556
5,1.084012,0.998970,0.472254
6,0.926297,0.724600,0.518219
7,0.781698,0.678215,0.654221
8,0.637948,0.530942,0.540171
9,0.518330,0.494679,0.661034
10,0.425887,0.470110,0.729167


Map:   0%|          | 0/115 [00:00<?, ? examples/s]

,type_voyage,vol,voiture
DL Fine-Tuning phase 1 — 3 graines,0.547 ± 0.015,0.602 ± 0.006,0.808 ± 0.029


In [ ]:
df3 = pd.DataFrame([{"approche": "DL Fine-Tuning phase 1 — moyenne 3 graines", "champ": c,
                     "F1_macro": round(float(np.mean(v)), 3), "ecart_type": round(float(np.std(v)), 3)}
                    for c, v in scores.items()])
pd.concat([df, df3]).to_csv("resultats/dl_classif.csv", index=False)

In [ ]:
# 1. Copier les résultats (instantané)
!cp -r resultats /content/drive/MyDrive/voyara_dl/
!ls /content/drive/MyDrive/voyara_dl/resultats

# 2. Vérifier que le modèle type_voyage est complet : les deux tailles doivent être identiques
!du -sh modeles/xlmr-voyara-type_voyage /content/drive/MyDrive/voyara_dl/modeles/xlmr-voyara-type_voyage

dl_classif.csv	dl_classif_historique.csv  dl_classif.md
1.1G	modeles/xlmr-voyara-type_voyage
1.1G	/content/drive/MyDrive/voyara_dl/modeles/xlmr-voyara-type_voyage


In [ ]:
!rm -rf /content/drive/MyDrive/voyara_dl/modeles/xlmr-voyara-type_voyage
!cp -r modeles/xlmr-voyara-type_voyage /content/drive/MyDrive/voyara_dl/modeles/

In [ ]:
!rm -rf /content/drive/MyDrive/voyara_dl/modeles/{type_voyage-0,type_voyage-8,vol-0,vol-8}

In [ ]:
!cp -r modeles/xlmr-voyara-vol modeles/xlmr-voyara-voiture /content/drive/MyDrive/voyara_dl/modeles/

## 5. Analyse critique (point 5) — à compléter
- Sur-apprentissage ? (courbes `trainer.state.log_history` : perte train vs validation)
- Classes confondues (voir la matrice) et pourquoi.
- Résultats par langue : la darija et l'arabe tiennent-ils ?
- Petit modèle fine-tuné vs LLM de l'étape 4 : vitesse (ms vs 15–25 s), coût, champs couverts.
- Limites : paraphrases générées, test de 115 demandes.

In [ ]:
# Exemples d'erreurs à montrer au prof
for e, p in zip(test, ft["type_voyage"][0]):
    if e["type_voyage"] != p: print(f"[{e['langue']}] attendu={e['type_voyage']:12s} prédit={p:12s} | {e['texte'][:110]}")